# 17: Estado estruturado (cada agente no seu campo)

Até aqui, o Supervisor decidia o próximo passo **contando mensagens pelo nome**. Havia uma mensagem do YFinance e
nenhuma do Analyzer? Etapa 2. O Consolidador fazia a mesma conta para escrever as limitações ("as notícias não foram buscadas").

Isso funciona enquanto cada agente fala exatamente uma vez e na ordem certa. Os passos mais avançados do projeto quebram
as duas condições:

- **Agentes em paralelo** (o `Send` do LangGraph): o Analyzer e o Investigator rodando ao mesmo tempo terminam em
  qualquer ordem, e a posição das mensagens deixa de dizer quem fez o quê.
- **Retry**: um agente que falha e roda de novo deixa mensagens de erro no histórico. Se o código contasse mensagens
  pelo nome, veria uma mensagem do agente e acharia que ele já entregou o trabalho.

A solução é dar a cada saída um lugar próprio no state. O nó do YFinance grava `dados_mercado`, a passagem única do
Analyzer grava simultaneamente `noticias` e `contexto_setor` (via `separar_secoes`), e o Investigator grava `diagnostico`.
Os guards e o Consolidador passam a perguntar "este campo está preenchido?", e não "quantas mensagens com este
nome existem?". Para quem usa, o pipeline fica muito mais previsível.

In [1]:
import sys
import warnings
from pathlib import Path

# Sobe pastas até achar a raiz do projeto (onde está o pyproject.toml), de onde quer que o notebook seja aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ))

from dotenv import load_dotenv
_ = load_dotenv(RAIZ / ".env")

# Aviso de depreciação interno do LangGraph, sem relação com o projeto: só polui a saída.
# O filtro vem depois do import do langchain_core, que instala o filtro dele ao ser importado.
from langchain_core._api.deprecation import LangChainPendingDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainPendingDeprecationWarning)
warnings.filterwarnings("ignore", message="Function wrap_gemini is in beta")

## 1. Conceitos

**State do LangGraph:** o dicionário que passa de nó em nó. Cada nó recebe o state inteiro e devolve só as chaves
que quer mudar. O LangGraph junta essa resposta ao state antes de chamar o próximo nó.

**TypedDict:** a classe que descreve o state (`MessagesState`, em `shared/types.py`). Ela diz quais chaves existem
e o tipo de cada uma, mas não obriga nenhuma a estar presente: no primeiro passo o state tem só `messages`. Por
isso todo acesso aos campos novos é `state.get(...)`, que devolve `None` quando o campo não existe.

**Reducer:** a regra que diz como juntar o valor novo com o antigo. Cada chave escolhe a sua:

| Chave | Reducer | Efeito |
|---|---|---|
| `messages` | `add_messages` | concatena: a mensagem nova entra no fim do histórico |
| `loop_count` | `operator.add` | soma: cada passagem do Supervisor devolve 1 |
| as demais (`dados_mercado`, `noticias`...) | nenhum (padrão) | sobrescreve: vale o último valor escrito |

**Por que `messages` continua:** os agentes ainda recebem o histórico inteiro como contexto (o Analyzer lê os
dados do YFinance nas mensagens), o Consolidador monta os insumos do LLM a partir delas e o dashboard mostra a
conversa. Os campos novos são uma cópia do texto final de cada agente, guardada num lugar fixo para **decidir**.

**Sinais de anomalia (o que aciona o Investigator):** regras em código sobre os números da empresa. Queda do lucro
líquido de mais de 30% contra o **mesmo trimestre do ano anterior** (comparar com o ano anterior tira o efeito da
sazonalidade), prejuízo no último trimestre e Piotroski F-Score de 3 ou menos (uma nota de 0 a 9 sobre a saúde
financeira: lucro, caixa, endividamento, eficiência). O notebook 10 mostra as regras rodando.

## 2. Os campos e seus reducers, lidos da própria classe

`get_type_hints` com `include_extras=True` mostra o `Annotated` de cada chave: quando há um reducer, ele aparece
ali. As chaves sem reducer usam o padrão, que sobrescreve.

In [2]:
import typing

from fluxo_financeiro.shared.types import MessagesState

dicas = typing.get_type_hints(MessagesState, include_extras=True)
for chave, tipo in dicas.items():
    extras = getattr(tipo, "__metadata__", ())
    reducer = extras[0].__name__ if extras else "(padrão: sobrescreve)"
    print(f"{chave:22} {reducer}")

messages               _add_messages
next_agent             (padrão: sobrescreve)
loop_count             add
supervisor_reasoning   (padrão: sobrescreve)
decidido_por           (padrão: sobrescreve)
sinais_anomalia        (padrão: sobrescreve)
relatorio              (padrão: sobrescreve)
dados_mercado          (padrão: sobrescreve)
noticias               (padrão: sobrescreve)
contexto_setor         (padrão: sobrescreve)
diagnostico            (padrão: sobrescreve)
erros                  or_
tentativas             or_


## 3. Sobrescrever x somar, num grafo de brinquedo

Um grafo mínimo, sem LLM, com dois nós que escrevem nas mesmas chaves. `contador` tem `operator.add`; `texto` não
tem reducer.

In [3]:
import operator
from typing import Annotated, Optional, TypedDict

from langgraph.graph import END, START, StateGraph


class Brinquedo(TypedDict):
    contador: Annotated[int, operator.add]
    texto: Optional[str]


def primeiro(state):
    return {"contador": 1, "texto": "escrito pelo primeiro nó"}


def segundo(state):
    print("o segundo nó leu:", state)
    return {"contador": 1, "texto": "escrito pelo segundo nó"}


g = StateGraph(Brinquedo)
g.add_node("primeiro", primeiro)
g.add_node("segundo", segundo)
g.add_edge(START, "primeiro")
g.add_edge("primeiro", "segundo")
g.add_edge("segundo", END)

print("estado final:  ", g.compile().invoke({"contador": 0}))

o segundo nó leu: {'contador': 1, 'texto': 'escrito pelo primeiro nó'}
estado final:   {'contador': 2, 'texto': 'escrito pelo segundo nó'}


## 4. O caso que enganava a contagem de mensagens

Um estado montado à mão em que o Analyzer falhou na primeira tentativa e deixou uma mensagem de erro no histórico.
Pela contagem de mensagens (`name="Analyzer"`), haveria 1 mensagem e a regra antiga acharia que a etapa do Analyzer
já tinha acontecido, pulando para a etapa seguinte.

Pela regra nova dos campos, `dados_mercado` está preenchido mas `noticias` é `None` (a falha não gravou dados).
O guard da Etapa 2 vê `noticias` vazio e aciona obrigatoriamente o Analyzer, sem consultar o LLM.

In [4]:
from langchain_core.messages import HumanMessage

from fluxo_financeiro.graph.supervisor import supervisor_node

pedido = HumanMessage("Analise a ação VALE3")
dados = HumanMessage("Dados para VALE3.SA: ...", name="YFinance")
erro_analyzer = HumanMessage("Erro: Analyzer falhou depois de 1 tentativa (transitorio): TimeoutError", name="Analyzer")

estado = {
    "messages": [pedido, dados, erro_analyzer],
    "loop_count": 1,
    "sinais_anomalia": [],
    "dados_mercado": dados.text,
    "noticias": None,
}


def contar_como_antes(messages):
    """Reconstrução da regra antiga, só para comparação: decidia pelo número de mensagens do Analyzer."""
    n = sum(1 for m in messages if m.name == "Analyzer")
    if n == 0:
        return "Analyzer (etapa 2)"
    return "etapa seguinte (achando que o Analyzer já entregou a análise)"


print("Mensagens do Analyzer no histórico:", sum(1 for m in estado["messages"] if m.name == "Analyzer"))
print("Regra antiga (contando mensagens):", contar_como_antes(estado["messages"]))

decisao = supervisor_node(estado)
print(f"Regra nova (campos do state):      {decisao['next_agent']} (decidido por: {decisao['decidido_por']})")
print("Tarefa:", decisao["messages"][0].text[:160], "...")

2026-11-08 22:00:02,673 - fluxo_financeiro - INFO - [supervisor] inicio — loop_count=1


2026-11-08 22:00:02,673 - fluxo_financeiro - INFO - [supervisor] guard: YFinance coletou → forcando Analyzer (notícias e setor)


Mensagens do Analyzer no histórico: 1
Regra antiga (contando mensagens): etapa seguinte (achando que o Analyzer já entregou a análise)
Regra nova (campos do state):      Analyzer (decidido por: guard)
Tarefa: Busque notícias recentes sobre a empresa no histórico. Use os dados financeiros do YFinance como contexto para identificar o ticker e o nome da empresa.
Busque  ...


`decidido por: guard` prova que o LLM não foi consultado: o guard leu `dados_mercado` preenchido e
`noticias` vazio e forçou o Analyzer. A contagem de mensagens teria sido enganada pelo erro e avançado indevidamente.

## 5. Uma análise real da WEGE3

O pipeline inteiro, uma vez (cerca de 30 segundos, algumas chamadas ao Gemini). O `stream` devolve o que cada nó
mudou; a cada decisão do Supervisor, a célula mostra quais campos já estavam preenchidos naquele momento e qual
regra decidiu.

In [5]:
from langchain_core.messages import HumanMessage

from fluxo_financeiro import app

CAMPOS = ("dados_mercado", "noticias", "contexto_setor", "diagnostico")


def regra(antes, update):
    """Qual condição sobre os campos explica a decisão (a mesma ordem dos guards no supervisor_node)."""
    if update["decidido_por"] == "hard_stop":
        return "loop_count chegou a 6"
    if update["decidido_por"] == "llm":
        return "nenhum guard se aplicou: o LLM decidiu"
    if update["next_agent"] == "FINISH":
        return "diagnostico preenchido ou encerramento"
    if update["next_agent"] == "Analyzer":
        return "dados_mercado preenchido, noticias vazio (etapa 2)"
    return "decisão do supervisor"


preenchidos = set()
final = None
for modo, pedaco in app.stream({"messages": [HumanMessage("Analise a ação WEGE3")]}, stream_mode=["updates", "values"]):
    if modo == "values":
        final = pedaco
        continue
    for no, update in pedaco.items():
        if no == "Supervisor":
            antes = sorted(preenchidos)
            print(f"Supervisor -> {update['next_agent']:12} por {update['decidido_por']:9} | campos: {antes or 'nenhum'}")
            print(f"   regra: {regra(antes, update)}")
        else:
            novos = [c for c in CAMPOS if update.get(c) is not None]
            preenchidos.update(novos)
            print(f"{no:12} gravou: {novos or 'nenhum campo de agente'}")

2026-11-08 22:00:02,684 - fluxo_financeiro - INFO - [supervisor] inicio — loop_count=0


2026-11-08 22:00:04,135 - fluxo_financeiro - INFO - [supervisor] decisao=YFinance | tarefa='Coletar dados financeiros e histórico de preços para WEGE3.'


2026-11-08 22:00:04,137 - fluxo_financeiro - DEBUG - [router] YFinance → YFinance


2026-11-08 22:00:04,138 - fluxo_financeiro - INFO - [node] YFinance: invocando agente (2 msgs)


Supervisor -> YFinance     por llm       | campos: nenhum
   regra: nenhum guard se aplicou: o LLM decidiu


2026-11-08 22:00:05,019 - fluxo_financeiro - INFO - [tool] buscar_dados_acao_b3: ticker=WEGE3.SA


... (24 outputs left out)

2026-11-08 22:00:23,689 - fluxo_financeiro - INFO - [node] Consolidador: consolidando (5 msgs)


Supervisor -> FINISH       por llm       | campos: ['contexto_setor', 'dados_mercado', 'noticias']
   regra: nenhum guard se aplicou: o LLM decidiu


2026-11-08 22:00:25,457 - fluxo_financeiro - INFO - [node] Consolidador: relatório pronto (1225 chars)


Consolidador gravou: nenhum campo de agente


## 6. Os quatro campos no state final

In [6]:
for campo in CAMPOS:
    valor = final.get(campo)
    if valor is None:
        print(f"{campo:15} None")
    else:
        print(f"{campo:15} {len(valor):5} caracteres | {valor[:90]!r}...")

print()
print("loop_count:", final["loop_count"])
print("Limitações do relatório:", final["relatorio"].limitacoes if final.get("relatorio") else "relatório de emergência")

dados_mercado    2586 caracteres | 'Dados para WEGE3.SA:\nEmpresa: WEG SA\nPreço Atual: R$ 48.19 (no dia: -0.23%)\nFaixa de 52 se'...
noticias          832 caracteres | 'A WEG (**WEGE3**) negocia atualmente a R$ 48.19, mantendo-se sólida com valor de mercado d'...
contexto_setor    900 caracteres | 'Inserida no subsetor de Máquinas e Equipamentos dentro de Bens Industriais, a WEG ocupa a '...
diagnostico     None

loop_count: 3
Limitações do relatório: []


## 7. Os campos gravam o texto exato de cada agente

O texto em cada campo é o mesmo que o agente produziu: para o YFinance e Investigator, a mensagem inteira; para
o Analyzer (em passagem única), cada campo recebe a sua seção fatiada por `separar_secoes` (`## Notícias da empresa`
e `## Contexto do setor`). A mensagem completa permanece no histórico.

In [7]:
from fluxo_financeiro.agents.analysts import separar_secoes

ultima = {}
for m in final["messages"]:
    if m.name in ("YFinance", "Analyzer", "Investigator"):
        ultima.setdefault(m.name, []).append(m.text)

noticias_esperadas, setor_esperado = separar_secoes(ultima["Analyzer"][0])

print("dados_mercado == mensagem do YFinance:", final["dados_mercado"] == ultima["YFinance"][0])
print("noticias == seção de notícias da mensagem:", final["noticias"] == noticias_esperadas)
print("contexto_setor == seção do setor da mensagem:", final["contexto_setor"] == setor_esperado)
if "Investigator" in ultima:
    print("diagnostico == mensagem do Investigator:", final["diagnostico"] == ultima["Investigator"][0])

dados_mercado == mensagem do YFinance: True
noticias == seção de notícias da mensagem: True
contexto_setor == seção do setor da mensagem: True


## O que observar

- As chaves de estado têm reducers específicos: `messages` (`add_messages`, concatena) e `loop_count` (`operator.add`, soma); os campos de conteúdo sobrescrevem diretamente.
- No estado com erro do Analyzer, a contagem antiga pularia a etapa achando que o agente já havia respondido. O guard novo leu `noticias` como `None` e manteve a obrigatoriedade da Etapa 2, decidido por guard sem LLM.
- Na WEGE3, o primeiro passo não tinha campo nenhum (o state começa só com `messages`): nenhum guard se aplicou e o LLM mandou para o YFinance.
- A Etapa 2 foi decidida por guard lendo `dados_mercado` sem `noticias`. O Analyzer gravou simultaneamente `noticias` e `contexto_setor` numa passagem só.
- A WEGE3 não teve sinal de anomalia, então o LLM escolheu FINISH na Etapa 4. Por isso `diagnostico` ficou `None` e o `loop_count` terminou em 3 (em vez de 4 na versão anterior).
- Os campos `dados_mercado` e `diagnostico` correspondem às mensagens integrais dos seus agentes; `noticias` e `contexto_setor` correspondem fielmente às seções extraídas por `separar_secoes`.
- Os números da análise mudam a cada execução. O que não muda é qual guard lê qual campo.